# Fine-tuning GENCO with a load-growth response term

`load_response` shows that the released GENCO under-reacts to system-wide load growth: when every load grows, its voltage
drops are 0.60 of the AC ones (median over 129 scenarios). This damping is why it glides past the fold and why its
warning is weak there. This notebook tests whether supervising that response during fine-tuning removes it.

It is `finetune.ipynb` with one addition, fixed here before any weight is changed:

- **Load-growth pairs.** Every training case that is a regression target is solved a second time with every load 2 %
  higher (loads only, the slack taking the extra power, as in the `load_response` probe). Every optimizer step adds one
  micro-batch of such pairs and penalises the difference between GENCO's voltage change per unit of load growth and the
  AC one, at the PQ buses: `step loss = published objective + w x response loss`. The weight `w` is the ratio of the
  released model's published-objective loss to its response loss on the pilot cases, so that at the start both
  contribute equally (capped at 100).

The training cases, their labels, the train/validation split, the validation set, the objective, the optimiser, the
number of epochs and the checkpoint selection (lowest validation loss) are those of `finetune.ipynb`; only the time
budget is raised so that the pairs still fit four epochs. The probe scenarios (N-2 blocks 0 and 1) and every evaluation
block are excluded from training, as in `finetune.ipynb`.

**Decision rule, fixed before the run.** The load-growth term is adopted only if, for seed 0, (a) the median load-growth
slope on the 129 `load_response` scenarios (`load_response_finetuned`) is at least 0.85 and above that of the published
fine-tuned model on the same scenarios, and (b) missed decisions per clustered frozen-test case fall against the released
model with the paired 95 % CI of the difference below zero (`evaluate_finetuned_loadgrowth`). Only then is seed 1 run.
Otherwise the published model stays as it is, and this run is reported as tried.

Set `TRAIN_SEED` (0 or 1) in the settings cell.

Runtime: T4 GPU, about 1.2 GB of downloads, 2.5 to 3.5 hours. Output: `finetune_loadgrowth_seed<TRAIN_SEED>.zip`
(about 80 MB with the checkpoint), saved automatically to `MyDrive/genco_runs`.


## Settings (frozen)

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
LOCAL_DIR = "genco-pf-contingency"
TRAIN_SEED = 0                        # training run: 0 or 1 (data, split and validation set do not change)
OUT_DIR = f"finetune_loadgrowth_seed{TRAIN_SEED}"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # pinned upstream code
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # pinned model and data revision (unchanged since 1 October 2026)

LEVEL_BASE = 2
TRAIN_PARTS = [2, 3, 4, 5]            # N-2 data blocks of the training operating points
EXCLUDED_PARTS = [0, 1, 6, 7, 16]     # storms, frozen test, forgetting check: never used here
BASE_PARTS = TRAIN_PARTS              # (the download cell downloads these)
N_CHECK = 16
TRAIN_FAMILIES = [("none", 2, "stress"),
                  ("uniform", 5, 1), ("uniform", 10, 1), ("uniform", 20, 1), ("uniform", 10, "stress"),
                  ("nearest", 10, 1), ("nearest", 20, 1), ("nearest", 10, "stress"), ("nearest", 20, "stress"),
                  ("corridor", 20, 1), ("corridor", 20, "stress")]          # (construction, total outages, load factor 1 or sampled)
LAM_RANGE = (0.85, 1.35)
HELD_OUT = ["footprint"]
LOW_V_EXCLUDE = 0.5                   # verified states with a voltage below this (suspect lower branch) are not used as targets
VAL_FRACTION = 0.10
LR, WEIGHT_DECAY, WARMUP_STEPS = 5e-5, 0.01, 20
BATCH, ACCUM, CLIP = 4, 4, 1.0
VAL_EVERY, VAL_MAX, VAL_BATCH = 100, 600, 8
PILOT_CASES = 64
TRAIN_BUDGET_MIN, MAX_EPOCHS = 105, 4   # 70 in finetune.ipynb; raised so the pairs still fit 4 epochs
POLICY = {"tol_pu": 1e-8, "max_iter": 30, "line_limit": 1.0, "voltage_buses": "pq", "low_voltage_flag": 0.5}
REGION_HOPS, FOOT_HOPS, FOOT_SCALE, BETWEENNESS_SAMPLES = 4, 6, 2.0, 400
TOL, MAX_NR = 1e-8, 30
MAX_DOWNLOAD_GB = 3
GROWTH = 0.02                         # load-growth pairs: every load 2 % higher (loads only, as in load_response)
PAIRS_PER_STEP = 4                    # pairs added to every optimizer step (one micro-batch of 2 x 4 graphs)
PAIR_VAL_MAX = 200                    # validation pairs used to monitor the response
MAX_RESPONSE_WEIGHT = 100.0
SEED = 0

In [ ]:
# Google Drive: results are copied to MyDrive/genco_runs at the end (Colab asks for permission once)
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/genco_runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("results will be saved to", DRIVE_DIR)
except Exception as e:  # not running on Colab, or Drive declined
    DRIVE_DIR = None
    print("Google Drive not available, results stay in the session:", e)


def save_to_drive(path):
    """Copies a result file to Google Drive and starts a browser download as well."""
    if DRIVE_DIR:
        shutil.copy(path, DRIVE_DIR)
        print(f"saved {os.path.basename(path)} to {DRIVE_DIR}")
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        pass


In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
import base64
GENCO_VERIFY_SOURCE = base64.b64decode('...').decode()
exec(GENCO_VERIFY_SOURCE)
print('genco_verify loaded; lightsim2grid (KLU):', HAVE_KLU)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""AC sensitivity helpers: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the shared power-flow helpers above."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""Load-sweep helpers: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC solution along lambda up to the largest factor at which a Newton start converges (the
"nose"), a check on that nose, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the shared power-flow helpers above and power_flow_jacobian from the AC sensitivity helpers."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor with an AC solution, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # no start converges at lambda = 1: go down until one does
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Expected to drop toward 0 at a fold."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


"""Extended verification helpers: generator reactive limits inside the AC truth and the check; stress patterns other
than the K1 scaling (loads only, reactive demand only, area-to-area transfer, generator outage on top of the storm);
warm-start timing of the classical solver. Uses LambdaCase / find_lambda_max / truth_below_nose from the load-sweep helpers and
build_ybus / newton / klu_solve / system_of / dc_angles / SN_MVA / HAVE_KLU from the shared power-flow helpers above."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import splu


# ================================================================ reactive limits
def bus_q_limits(d, gen_rows):
    """Per-bus reactive limits (p.u.) from the generator table of one scenario: sums over in-service generators.
    gen_rows: DataFrame with bus, min_q_mvar, max_q_mvar, in_service, is_slack_gen. Buses without a generator get 0/0."""
    nb = d["nb"]
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    g = gen_rows[gen_rows["in_service"].to_numpy() > 0.5] if "in_service" in gen_rows else gen_rows
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float) / SN_MVA)
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float) / SN_MVA)
    return qmin, qmax


def q_generation(Y, V, Qd_pu):
    """Reactive generation implied by a voltage state (p.u.): injected Q plus the reactive load."""
    return (V * np.conj(Y @ V)).imag + Qd_pu


class QLimCase(LambdaCase):
    """A LambdaCase whose solver enforces generator reactive limits: a PV bus whose generation would exceed its limit
    is held at the limit as a PQ bus (and released again if its voltage recovers past the set-point). The outer loop
    is the standard one used in power-flow programs; a case counts as solved only when no further switching is needed."""

    def __init__(self, d, qmin_pu, qmax_pu, max_outer=15, eps=1e-6):
        super().__init__(d)
        self.qmin, self.qmax, self.max_outer, self.eps = qmin_pu, qmax_pu, max_outer, eps
        self.vset = d["Vm"].copy()                      # generator set-points (the base solution holds them)
        self.limited = self.PV & (self.qmax >= self.qmin)
        self.last = {"switched": {}, "outer": 0, "iters": 0}

    def _inner(self, S, V0, pv, pq, max_iter, tol):
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, pv, pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson with reactive-limit switching. Returns V, converged, total inner iterations, seconds.
        `self.last` keeps the switched buses of the last call ({bus: 'max' | 'min'}) and the number of outer rounds."""
        t0 = time.perf_counter()
        S_base = self.S(lam)
        Qd = lam * self.Qd
        V = np.asarray(V0, complex).copy()
        switched, backs, total_it = {}, {}, 0
        for outer in range(1, self.max_outer + 1):
            sw = np.array(sorted(switched), dtype=int)
            pv = np.array([b for b in self.pv if b not in switched], dtype=int)
            pq = np.array(sorted(set(self.pq.tolist()) | set(switched)), dtype=int)
            S = S_base.copy()
            if len(sw):
                S[sw] = S_base.real[sw] + 1j * (np.array([self.qmax[b] if switched[b] == "max" else self.qmin[b] for b in sw]) - Qd[sw])
            V[pv] = self.vset[pv] * np.exp(1j * np.angle(V[pv]))      # PV buses start at their set-point
            V, ok, it, _ = self._inner(S, V, pv, pq, max_iter, tol)
            total_it += it
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, False, total_it, time.perf_counter() - t0
            Qg = q_generation(self.Y, V, Qd)
            to_max = [b for b in pv if self.limited[b] and Qg[b] > self.qmax[b] + self.eps]
            to_min = [b for b in pv if self.limited[b] and Qg[b] < self.qmin[b] - self.eps]
            Vm = np.abs(V)
            back = [b for b in sw if backs.get(b, 0) < 2 and ((switched[b] == "max" and Vm[b] > self.vset[b] + self.eps) or
                                                             (switched[b] == "min" and Vm[b] < self.vset[b] - self.eps))]
            if not to_max and not to_min and not back:
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, True, total_it, time.perf_counter() - t0
            for b in to_max:
                switched[int(b)] = "max"
            for b in to_min:
                switched[int(b)] = "min"
            for b in back:
                del switched[int(b)]; backs[int(b)] = backs.get(int(b), 0) + 1
        self.last = {"switched": dict(switched), "outer": self.max_outer, "iters": total_it}
        return V, False, total_it, time.perf_counter() - t0

    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        """As LambdaCase.decisions, but voltage limits are checked at every non-slack bus (a generator at its limit no
        longer holds its voltage) and the reactive generation beyond the limits is reported."""
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        chk = ~self.REF
        low = chk & (Vm < self.vmin + margin_v)
        high = chk & (Vm > self.vmax - margin_v)
        vm_chk = np.where(chk, Vm, np.inf)
        Qg = q_generation(self.Y, V, lam * self.Qd)
        over_q = self.limited & (Qg > self.qmax + 1e-6)
        return {"min_vm": float(vm_chk.min()), "weak_bus": int(np.argmin(vm_chk)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA),
                "n_gen_over_qmax": int(over_q.sum()), "q_over_qmax_Mvar": float(SN_MVA * np.clip(Qg - self.qmax, 0, None)[self.limited].sum())}


# ================================================================ stress patterns
class StressCase(LambdaCase):
    """A storm topology stressed in one direction by a factor lambda (1 = the actual operating point):
    'all'      every load and every non-slack generator scaled together (the K1 pattern; slack takes the rest);
    'load'     loads only (P and Q at constant power factor); generation fixed, the slack takes the growth;
    'reactive' reactive demand only (Q of every load); active power unchanged;
    'transfer' loads inside an area scaled, generation outside the area scaled up by the same MW (a transfer into
               the storm area); the slack takes the losses."""

    def __init__(self, d, mode="all", area=None, kappa=None):
        super().__init__(d)
        self.mode = mode
        self.Pg, self.Pd = d["Pg"] / SN_MVA, d["Pd"] / SN_MVA
        self.area = np.zeros(self.nb, bool) if area is None else np.asarray(area, bool)
        self.genB = (self.Pg > 0) & ~self.area & ~self.REF
        self.kappa = kappa if kappa is not None else (self.Pd[self.area].sum() / max(self.Pg[self.genB].sum(), 1e-9))

    def S(self, lam):
        if self.mode == "all":
            return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "load":
            return (self.Pg - lam * self.Pd) + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "reactive":
            return self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "transfer":
            fl = 1.0 + (lam - 1.0) * self.area
            fg = 1.0 + (lam - 1.0) * self.kappa * self.genB
            return (self.Pg * fg - self.Pd * fl) + 1j * (self.Qg - self.Qd * fl)
        raise ValueError(self.mode)

    def dc_loading(self, lam):
        """DC loading at this lambda: angles from the scaled active injections through the DC susceptance matrix."""
        d = self.d
        nb, on = d["nb"], d["on"]
        f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
        B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
        keep = self.nonref
        th = np.zeros(nb)
        th[keep] = splu(B[keep][:, keep]).solve(self.S(lam).real[keep])
        return self.loading(np.exp(1j * (th + np.radians(float(d["Va"][self.ref[0]])))))


def storm_area(d, footprint_buses, min_load_share=0.10, max_hops=6):
    """Buses within h hops of the storm footprint, h grown until the area holds at least `min_load_share` of the load."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=np.asarray(footprint_buses, int), min_only=True)
    total = max(d["Pd"].sum(), 1e-9)
    for h in range(0, max_hops + 1):
        area = hop <= h
        if d["Pd"][area].sum() >= min_load_share * total:
            return area, h, hop
    return hop <= max_hops, max_hops, hop


def footprint_of(d, removed_rows):
    """Endpoint buses of the removed lines (branch-row indices)."""
    return np.unique(np.r_[d["f"][removed_rows], d["t"][removed_rows]])


def pick_generator_outage(d, gen_rows, hop, max_hops=3):
    """The largest in-service non-slack generator (by active output; by reactive range if none produces active power)
    whose bus lies within `max_hops` of the storm footprint; the largest anywhere if there is none. Returns its row
    label in gen_rows (or None)."""
    g = gen_rows
    keep = np.ones(len(g), bool)
    if "in_service" in g:
        keep &= g["in_service"].to_numpy(float) > 0.5
    if "is_slack_gen" in g:
        keep &= g["is_slack_gen"].to_numpy(float) < 0.5
    keep &= ~d["REF"][g["bus"].to_numpy().astype(int)]
    g = g[keep]
    if len(g) == 0:
        return None
    size = g["p_mw"].to_numpy(float)
    if not (size > 0).any():
        size = g["max_q_mvar"].to_numpy(float) - g["min_q_mvar"].to_numpy(float)
    near = hop[g["bus"].to_numpy().astype(int)] <= max_hops
    cand = g[near] if near.any() else g
    size_c = size[near] if near.any() else size
    return int(cand.index[int(np.argmax(size_c))])


def remove_generator(d, gen_rows, row):
    """The truth model after one generator is lost: its P and Q leave the bus (the slack takes the P); if no in-service
    generator remains at the bus, it becomes a PQ bus. Returns the new dict and the reduced generator table."""
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    g = gen_rows.copy()
    bus = int(g.loc[row, "bus"])
    d2["Pg"][bus] -= float(g.loc[row, "p_mw"]); d2["Qg"][bus] -= float(g.loc[row, "q_mvar"])
    g = g.drop(index=row)
    left = g[(g["bus"].to_numpy().astype(int) == bus) & (g["in_service"].to_numpy() > 0.5)] if "in_service" in g else g[g["bus"] == bus]
    if len(left) == 0 and not d2["REF"][bus]:
        d2["PV"][bus] = False; d2["PQ"][bus] = True
        d2["Pg"][bus] = 0.0; d2["Qg"][bus] = 0.0
    return d2, g, bus


# ================================================================ warm starts (item 9)
def timed_solves(case, lam, starts, max_iter=30, tol=1e-8, reps=3):
    """Iterations and the median time (ms) of the solver from each start; converged flag per start."""
    out = {}
    for name, V0 in starts:
        its, ts, oks = [], [], []
        for _ in range(reps):
            V, ok, it, dt = case.solve(lam, V0, max_iter, tol)
            its.append(it); ts.append(1000 * dt); oks.append(bool(ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2))
        out[name] = {"converged": all(oks), "iters": int(np.median(its)), "ms": float(np.median(ts))}
    return out


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from gridfm_graphkit.datasets.globals import PD_H, QD_H, PG_H

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# N-2 scenarios that are hard for GENCO: not used as storm base points, so the base points are ordinary
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def pick_storm(d, extra, rng):
    """`extra` more lines to remove, the nearest (in hops) to a random centre bus, keeping the grid in one piece.
    Returns branch-row indices, the storm radius in hops and the centre bus; (None, None, centre) if impossible."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None, center
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)


def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}



import subprocess, math

## 1. The frozen protocol

In [ ]:
# Everything that decides the fine-tuned model is fixed here, before any case is built or any weight is changed.
FT_PROTOCOL = {
    "purpose": "fine-tune the released GENCO checkpoint as in finetune.ipynb, with a load-growth response term, and evaluate it later "
               "(evaluate_finetuned_loadgrowth, load_response_finetuned) against the released and the published fine-tuned model",
    "differences_from_finetune": {
        "pairs": f"every regression target solved again with every load x (1 + {GROWTH}) (loads only; generators unchanged; the slack "
                 "takes the extra power), with the same solver policy and the same target criteria; a pair is used only if both states are targets",
        "response_loss": f"mean over the PQ buses of a micro-batch of {PAIRS_PER_STEP} pairs of ((GENCO Vm change - AC Vm change) / {GROWTH})^2",
        "weight": f"step loss = published objective (as in finetune.ipynb) + w x response loss, w = (released-model published-objective "
                  f"loss on the pilot cases) / (released-model response loss on as many pairs), capped at {MAX_RESPONSE_WEIGHT}",
        "budget": f"TRAIN_BUDGET_MIN = {TRAIN_BUDGET_MIN} (70 in finetune.ipynb), so that the same MAX_EPOCHS fit",
        "decision_rule": "adopted only if, for seed 0, (a) the median load-growth slope on the 129 load_response scenarios is >= 0.85 "
                         "and above the published fine-tuned model's, and (b) missed decisions per clustered frozen-test case fall "
                         "against the released model with the paired 95 % CI below zero"},
    "model": {"repo": REPO_ID, "checkpoint": MODEL_DIR, "graphkit_branch": GRAPHKIT_BRANCH, "precision": "float32"},
    "objective": "the released training objective, unchanged: graphkit MixedLoss of caseTexas_train.yaml = 0.9 x masked bus MSE (Vm, Va) "
                 "+ 0.1 x layered weighted physics residual (base weight 0.5), in the released normalisation",
    "data": {"level": LEVEL_BASE, "train_partitions": TRAIN_PARTS, "excluded_partitions": EXCLUDED_PARTS,
             "excluded_why": "0-1: base points of K1/N1/N2 (the storms of T2's near-fold test); 6, 16: the final evaluation F1 (T2's main test); "
                             "7: T2's forgetting check on released N-2 scenarios",
             "families": TRAIN_FAMILIES, "load_factor_range": LAM_RANGE, "held_out_constructions": HELD_OUT,
             "labels": "Newton-Raphson (lightsim2grid KLU) to 1e-8 p.u. from the pre-contingency, DC and flat starts with the branch check on; "
                       "a case is a regression target only if verified, with one solution among the starts and a lowest voltage >= LOW_V_EXCLUDE; "
                       "unresolved cases are kept as labels for the solvability head (notebook T2)",
             "split": f"by base point: {VAL_FRACTION:.0%} of the training base points (seeded) are validation, never trained on"},
    "optimiser": {"type": "AdamW", "lr": LR, "betas": [0.9, 0.999], "weight_decay": WEIGHT_DECAY, "warmup_steps": WARMUP_STEPS,
                  "batch": BATCH, "accumulation": ACCUM, "effective_batch": BATCH * ACCUM, "grad_clip": CLIP,
                  "schedule": "linear warm-up, then constant"},
    "selection": f"the checkpoint with the lowest validation loss (the training objective on the validation split), evaluated every {VAL_EVERY} "
                 "optimizer steps and at the end of each epoch; the released model's own validation loss is the starting point",
    "sizing": f"EPOCHS = min({MAX_EPOCHS}, max(1, floor({TRAIN_BUDGET_MIN} min / (pilot seconds per case x training cases)))); the pilot's weight "
              "updates are discarded",
    "evaluation": "notebook T2, frozen before this run: paired released vs fine-tuned on the 870 F1 cases (GENCO-alone misses against the "
                  "verified reference, worst-bus voltage error, Newton iterations, verdicts), on soft-footprint N-20 cases built on the F1 base "
                  "points (a construction held out of training), on released N-2 (block 7), N-10 and N-20 scenarios (forgetting), on the "
                  "120 K1 storms against their stored AC truth along the load sweep (near-fold response), and the solvability head",
    "seed": SEED, "train_seed": TRAIN_SEED,
}
json.dump(FT_PROTOCOL, open(Path(OUT_DIR) / "frozen_protocol.json", "w"), indent=1, default=str)
print(json.dumps({k: FT_PROTOCOL[k] for k in ["data", "optimiser", "selection", "sizing"]}, indent=1, default=str))

## 2. Download of the model and the training data blocks; the operating points

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
for p, s in all_files:
    m = RAW3_PAT.search(p) if p.startswith(f"data/test_{LEVEL_BASE}/") else None
    if m and int(m.group(2)) in BASE_PARTS:
        plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

In [ ]:
# The training operating points: every N-2 scenario of the training partitions (none of the excluded partitions is on disk)
import pyarrow.dataset as ds_
bus_path = find_local(LOCAL_DIR, LEVEL_BASE, "bus_data")
tab = ds_.dataset(str(bus_path), format="parquet", partitioning="hive").to_table(columns=["scenario", "scenario_partition"]).to_pandas().drop_duplicates()
PART_OF = dict(zip(tab.scenario.astype(int), tab.scenario_partition.astype(int)))
assert not set(PART_OF.values()) & set(EXCLUDED_PARTS), "an excluded partition is on disk"
BASE_IDS = np.array(sorted(s for s, p_ in PART_OF.items() if p_ in TRAIN_PARTS), dtype=np.int64)
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
NB = next(iter(BASE.values()))["nb"]
rng_split = np.random.default_rng(SEED)
VAL_BASES = set(int(x) for x in rng_split.choice(BASE_IDS, int(round(VAL_FRACTION * len(BASE_IDS))), replace=False))
print(f"{len(BASE_IDS)} training operating points ({NB} buses) in partitions {sorted(set(PART_OF[int(s)] for s in BASE_IDS))}; "
      f"{len(VAL_BASES)} of them are validation")

## 3. GENCO, and a check of the model

In [ ]:
# [GPU] build graphs of the base points (used for the model check)

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

BUS_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"), BASE_IDS)
GEN_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"), BASE_IDS)
BR_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data"), BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")


def build_graph(sid, base, Pd=None, Qd=None, removed_lines=()):
    """GENCO's input graph for base point `base` with extra lines removed and (optionally) other loads."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    if Pd is not None:
        b["Pd"], b["Qd"] = np.asarray(Pd, float), np.asarray(Qd, float)
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    return TRANSFORM(g)


def scaled_graph(g, lam):
    """The same graph with every load and every non-slack generator scaled by lam (the slack's generation is a
    masked input, 0, and stays 0; loads and generation are divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    return h


BASE_GRAPHS = [build_graph(int(s), int(s)) for s in BASE_IDS[:N_CHECK]]
print(f"{len(BASE_GRAPHS)} base graphs built")

In [ ]:
# [GPU] load model

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)

In [ ]:
# [GPU] run GENCO on a list of graphs; check the model against the stored AC solutions of the base points

def run_genco(graphs, bs=1):
    """GENCO's (Vm, Va in radians) for each graph, in order."""
    out = [None] * len(graphs)
    for i, h in enumerate(graphs):
        h["scenario_id"] = torch.tensor([i], dtype=torch.long)
    for b in DataLoader(graphs, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad():
            o = forward(MODEL, b)["bus"].float()
        ptr = b["bus"].ptr.cpu().tolist()
        for g, i in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            out[int(i)] = (o[ptr[g]:ptr[g + 1], VM_OUT].cpu().numpy().astype(np.float64),
                           o[ptr[g]:ptr[g + 1], VA_OUT].cpu().numpy().astype(np.float64))
    return out


t0 = time.perf_counter()
BASE_OUT = run_genco(BASE_GRAPHS[:N_CHECK])
T_FWD_MS = 1000 * (time.perf_counter() - t0) / len(BASE_OUT)
errs = np.array([np.max(np.abs(vm - BASE[int(s)]["Vm"])[BASE[int(s)]["PQ"]]) for s, (vm, va) in zip(BASE_IDS[:N_CHECK], BASE_OUT)])
print(f"GENCO on {len(BASE_OUT)} base points: worst-bus voltage error median {np.median(errs):.4f} p.u., "
      f"95th percentile {np.percentile(errs, 95):.4f} p.u. (expected about 0.001 and 0.004) | {T_FWD_MS:.0f} ms per graph incl. loading")
if np.median(errs) > 0.02 and NB > 100:
    raise RuntimeError("GENCO's error on ordinary N-2 cases is far above the known level: the model did not load correctly; check the installation and the downloaded checkpoint.")

## 4. The training cases and their verified labels

In [ ]:
# The contingency constructions of frozen-test / construction (uniform, nearest lines, corridor, soft footprint), unchanged
import networkx as nx
d0 = next(iter(BASE.values()))
G = nx.Graph()
for f_, t_ in zip(d0["f"][d0["on"]], d0["t"][d0["on"]]):
    G.add_edge(int(f_), int(t_))
t0 = time.perf_counter()
EB = nx.edge_betweenness_centrality(G, k=min(BETWEENNESS_SAMPLES, G.number_of_nodes()), seed=SEED, normalized=True)
LINE_BTW = np.array([EB.get((int(f_), int(t_)), EB.get((int(t_), int(f_)), 0.0)) for f_, t_ in zip(d0["f"], d0["t"])])
print(f"edge betweenness of the grid in {time.perf_counter() - t0:.0f} s")


def pick_lines(d, extra, mode, rng):
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    centre = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=centre)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    if mode == "nearest":
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    elif mode == "uniform":
        order = rng.permutation(cand)
    elif mode == "corridor":
        h = REGION_HOPS
        while (dist <= h).sum() < 3 * extra and h < 12:
            h += 1
        reg = cand[dist <= h]
        order = reg[np.lexsort((rng.random(len(reg)), -LINE_BTW[reg]))]
    elif mode == "footprint":
        sel = dist <= FOOT_HOPS
        reg, w = cand[sel], np.exp(-dist[sel] / FOOT_SCALE)
        keys = rng.random(len(reg)) ** (1.0 / w)
        order = reg[np.argsort(-keys)]
    else:
        raise ValueError(mode)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, np.nan, centre
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if removed else 0.0
    return np.array(removed, dtype=int), radius, centre


def scaled_dict(d, rows, lam):
    """The case with extra lines out and every load and every non-slack generator scaled by lam (as LambdaCase in K1)."""
    d2 = {kk: (v.copy() if isinstance(v, np.ndarray) else v) for kk, v in d.items()}
    if len(rows):
        d2["on"][rows] = False
    d2.pop("Va_dc", None)
    d2["Pd"] = d2["Pd"] * lam; d2["Qd"] = d2["Qd"] * lam
    d2["Pg"] = np.where(d2["REF"], d2["Pg"], d2["Pg"] * lam)
    return d2


def build_train_cases(base_ids):
    cases, skipped = [], 0
    for b_id in base_ids:
        d = BASE[int(b_id)]
        rng = np.random.default_rng(SEED * 1000003 + int(b_id) + 7)
        for j, (mode, k, stress) in enumerate(TRAIN_FAMILIES):
            lam = float(rng.uniform(*LAM_RANGE)) if stress == "stress" else 1.0
            if mode == "none":
                rem = np.array([], dtype=int)
            else:
                extra = k - int((~d["on"]).sum())
                rem, radius, centre = pick_lines(d, extra, mode, rng) if extra > 0 else (np.array([], dtype=int), 0.0, -1)
                if rem is None:
                    skipped += 1; continue
            fam = "load-only" if mode == "none" else f"{mode}-N{k}"
            cases.append({"cid": int(b_id) * 100 + j, "base": int(b_id), "family": fam, "mode": mode, "k": k, "stressed": stress == "stress",
                          "lam": lam, "rows": rem, "removed_lines": d["line"][rem].astype(int).tolist(),
                          "split": "val" if int(b_id) in VAL_BASES else "train"})
    return cases, skipped


CASES, skipped = build_train_cases(BASE_IDS)
print(f"{len(CASES)} training cases built ({skipped} could not be built); by family: "
      f"{pd.Series([c['family'] + (' stressed' if c['stressed'] else '') for c in CASES]).value_counts().to_dict()}")

In [ ]:
# Every case solved by the verification module (classical starts, branch check on); the regression targets are the verified,
# unambiguous states. Unresolved cases are kept as labels for the solvability head.
LABEL_POLICY = dict(POLICY, branch_check=True, branch_choice="first")
LABELS, rows = {}, []
t0 = time.perf_counter()
for i, c in enumerate(CASES):
    d2 = scaled_dict(BASE[c["base"]], c["rows"], c["lam"])
    ver = Verifier(d2, LABEL_POLICY)
    r = ver.run(policy_name="classical")
    multi = bool(r.flags.get("multiple_solutions", False))
    use = r.converged and not multi and r.limits["min_vm_all_buses"] >= LOW_V_EXCLUDE
    if use:
        V = np.array(r.state["vm"]) * np.exp(1j * np.radians(np.array(r.state["va_deg"])))
        qinj = (V * np.conj(ver.Y @ V)).imag * SN_MVA + d2["Qd"]            # reactive generation per bus, Mvar
        LABELS[c["cid"]] = (np.asarray(r.state["vm"], np.float32), np.asarray(r.state["va_deg"], np.float32), qinj.astype(np.float32))
    rows.append({"cid": c["cid"], "base": c["base"], "family": c["family"], "mode": c["mode"], "k": c["k"], "stressed": c["stressed"],
                 "lam": c["lam"], "removed_lines": json.dumps(c["removed_lines"]), "split": c["split"], "status": r.status,
                 "start_used": r.start_used, "iterations": r.iterations, "min_vm_all": r.limits.get("min_vm_all_buses"),
                 "n_solutions": r.flags.get("n_solutions_found", 0), "multiple_solutions": multi, "regression_target": bool(use)})
    if (i + 1) % 1000 == 0:
        print(f"  {i + 1} cases labelled ({time.perf_counter() - t0:.0f} s)")
# load-growth pairs: every regression target again with every load 2 % higher (loads only)
PAIR_LABELS, prow = {}, []
CASE_OF = {c["cid"]: c for c in CASES}
t1 = time.perf_counter()
for i, cid in enumerate(list(LABELS)):
    c = CASE_OF[cid]
    d2 = scaled_dict(BASE[c["base"]], c["rows"], c["lam"])
    d2["Pd"] = d2["Pd"] * (1 + GROWTH); d2["Qd"] = d2["Qd"] * (1 + GROWTH)
    ver = Verifier(d2, LABEL_POLICY)
    r = ver.run(policy_name="classical")
    multi = bool(r.flags.get("multiple_solutions", False))
    use = r.converged and not multi and r.limits["min_vm_all_buses"] >= LOW_V_EXCLUDE
    if use:
        V = np.array(r.state["vm"]) * np.exp(1j * np.radians(np.array(r.state["va_deg"])))
        qinj = (V * np.conj(ver.Y @ V)).imag * SN_MVA + d2["Qd"]
        PAIR_LABELS[cid] = (np.asarray(r.state["vm"], np.float32), np.asarray(r.state["va_deg"], np.float32), qinj.astype(np.float32))
    prow.append({"cid": cid, "split": c["split"], "family": c["family"], "stressed": c["stressed"], "status": r.status, "pair_used": bool(use)})
    if (i + 1) % 1000 == 0:
        print(f"  {i + 1} pairs labelled ({time.perf_counter() - t1:.0f} s)")
PC = pd.DataFrame(prow); PC.to_csv(Path(OUT_DIR) / "pair_cases.csv", index=False)
print(f"load-growth pairs: {int(PC.pair_used.sum())} of {len(PC)} targets also solved at +{GROWTH:.0%} load ({time.perf_counter() - t1:.0f} s)")
TC = pd.DataFrame(rows)
TC.to_csv(Path(OUT_DIR) / "train_cases.csv", index=False)
print(f"labelled {len(TC)} cases in {time.perf_counter() - t0:.0f} s")
display(TC.groupby(["family", "stressed"]).agg(cases=("cid", "count"), unresolved=("status", lambda x: int((x == "unresolved").sum())),
                                                multiple_solutions=("multiple_solutions", "sum"), targets=("regression_target", "sum")))

In [ ]:
# A training graph is GENCO's input graph of the case (exactly as in `storms`: the base point's graph with the lines out, then the loads
# and generation scaled) with the verified state written into the bus table, so that the targets are the verified Vm and Va.
from torch_geometric.data import Batch


def train_graph(c, label=None):
    b = BG.get_group(c["base"]).sort_values("bus").reset_index(drop=True).copy()
    if label is not None:
        vm, va, qg = label
        b["Vm"] = vm.astype(np.float64); b["Va"] = va.astype(np.float64)
        gen_bus = ((b["PV"] > 0.5) | (b["REF"] > 0.5)).to_numpy()
        b.loc[gen_bus, "Qg"] = qg[gen_bus].astype(np.float64)
    r = RG.get_group(c["base"]).copy()
    if len(c["removed_lines"]):
        r.loc[r["idx"].astype(int).isin([int(x) for x in c["removed_lines"]]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(c["cid"]), b, GG.get_group(c["base"]), r)
    NORMALIZER.transform(g)
    g = TRANSFORM(g)
    if c["lam"] != 1.0:
        g = scaled_graph(g, c["lam"])
    g["scenario_id"] = torch.tensor([0], dtype=torch.long)
    return g


def to_batch(graphs):
    return Batch.from_data_list(graphs).to(DEVICE)


CBY = {c["cid"]: c for c in CASES}


def pair_graph(c, label):
    """The training graph of case c with every load 2 % higher (loads are divided by baseMVA, so scaling the inputs is exact)."""
    g = train_graph(c, label)
    g["bus"].x = g["bus"].x.clone()
    g["bus"].x[:, PD_H] *= 1 + GROWTH; g["bus"].x[:, QD_H] *= 1 + GROWTH
    return g


# check 1: with the released state written back, the training graph of a base point is the graph GENCO is evaluated on
for c in [c for c in CASES if c["mode"] == "none"][:3]:
    d = BASE[c["base"]]
    c1 = dict(c, lam=1.0)
    g_a = train_graph(c1, (d["Vm"].astype(np.float32), d["Va"].astype(np.float32), d["Qg"].astype(np.float32)))
    g_b = build_graph(c["base"], c["base"])
    for key in ["bus", "gen"]:
        assert torch.allclose(g_a[key].x, g_b[key].x, atol=1e-4) and torch.allclose(g_a[key].y, g_b[key].y, atol=1e-4), f"graph mismatch ({key})"
# check 2: no label leaks into GENCO's input (the output with the true state written in equals the output with a dummy state)
MODEL.eval()
for cid in list(LABELS)[:3]:
    c = CBY[cid]; vm, va, qg = LABELS[cid]; d = BASE[c["base"]]
    gen_bus = d["PV"] | d["REF"]
    dummy = (np.where(d["PQ"], 1.0, vm).astype(np.float32), np.where(d["REF"], va, 0.0).astype(np.float32), np.where(gen_bus, 0.0, qg).astype(np.float32))
    g1, g2 = train_graph(c, LABELS[cid]), train_graph(c, dummy)
    # the inputs GENCO sees (after masking) must be bit-identical; only the targets may differ
    for key in ["bus", "gen"]:
        assert torch.equal(g1[key].x, g2[key].x), f"the target leaks into the {key} input"
    for et in g1.edge_types:
        if "edge_attr" in g1[et]:
            assert torch.equal(g1[et].edge_attr, g2[et].edge_attr), f"the target leaks into the {et} edge input"
    assert not torch.equal(g1["bus"].y, g2["bus"].y), "the targets were not written"
    with torch.no_grad():                    # outputs agree up to GPU non-determinism of the scatter sums
        o1 = forward(MODEL, to_batch([g1]))["bus"]; o2 = forward(MODEL, to_batch([g2]))["bus"]
    print(f"  case {cid}: inputs identical; largest output difference {float((o1 - o2).abs().max()):.1e} (GPU non-determinism)")
# check 3: a pair graph differs from its training graph only in the loads, by exactly 2 %
for cid in [c for c in PAIR_LABELS][:3]:
    c = CBY[cid]; g1, g2 = train_graph(c, LABELS[cid]), pair_graph(c, PAIR_LABELS[cid])
    other = [j for j in range(g1["bus"].x.shape[1]) if j not in (PD_H, QD_H)]
    assert torch.equal(g1["bus"].x[:, other], g2["bus"].x[:, other]) and torch.equal(g1["gen"].x, g2["gen"].x), "pair graph differs beyond the loads"
    assert torch.allclose(g2["bus"].x[:, PD_H], g1["bus"].x[:, PD_H] * (1 + GROWTH)), "loads not scaled by 1 + GROWTH"
print("checks passed: training graphs equal the evaluation graphs at the released state, the targets do not leak into the inputs, "
      "and pair graphs differ only in the loads")

## 5. Pilot, sizing and fine-tuning

In [ ]:
TRAIN_IDS = [cid for cid in LABELS if CBY[cid]["split"] == "train"]
VAL_IDS_ALL = [cid for cid in LABELS if CBY[cid]["split"] == "val"]
rng_v = np.random.default_rng(SEED + 1)
VAL_IDS = sorted(rng_v.choice(VAL_IDS_ALL, min(VAL_MAX, len(VAL_IDS_ALL)), replace=False).tolist())
VAL_GROUP = {cid: ("stressed" if CBY[cid]["stressed"] else ("clustered" if CBY[cid]["mode"] in ("nearest", "corridor") else "uniform/base")) for cid in VAL_IDS}
print(f"regression targets: {len(TRAIN_IDS)} training, {len(VAL_IDS_ALL)} validation ({len(VAL_IDS)} used for selection)")
PAIR_TRAIN = [cid for cid in TRAIN_IDS if cid in PAIR_LABELS]
PAIR_VAL = sorted(np.random.default_rng(SEED + 4).choice([cid for cid in VAL_IDS_ALL if cid in PAIR_LABELS],
                  min(PAIR_VAL_MAX, sum(cid in PAIR_LABELS for cid in VAL_IDS_ALL)), replace=False).tolist())
print(f"load-growth pairs: {len(PAIR_TRAIN)} for training, {len(PAIR_VAL)} for monitoring")


def response_loss(cids):
    """Mean over the PQ buses of ((GENCO Vm change - AC Vm change) / GROWTH)^2 for a micro-batch of pairs."""
    vb = forward(MODEL, to_batch([train_graph(CBY[c], LABELS[c]) for c in cids]))["bus"][:, VM_OUT].float()
    vg = forward(MODEL, to_batch([pair_graph(CBY[c], PAIR_LABELS[c]) for c in cids]))["bus"][:, VM_OUT].float()
    true = torch.tensor(np.concatenate([(PAIR_LABELS[c][0] - LABELS[c][0]) / GROWTH for c in cids]), device=vb.device)
    pq = torch.tensor(np.concatenate([BASE[CBY[c]["base"]]["PQ"] for c in cids]), device=vb.device)
    return (((vg - vb) / GROWTH - true)[pq] ** 2).mean()


def response_step(cids, weight):
    loss = response_loss(cids)
    (loss * weight).backward()
    return float(loss.detach())
import copy
RELEASED_STATE = copy.deepcopy({k: v.detach().cpu().clone() for k, v in MODEL.state_dict().items()})
opt = torch.optim.AdamW(MODEL.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=WEIGHT_DECAY)


def train_step(cids, weight):
    """One micro-batch: its loss is weighted by its share of the cases in the optimizer step it belongs to, so that every case
    counts equally whatever the micro-batch and step sizes (a short last micro-batch or a short last step of an epoch)."""
    out, ld = TASK.shared_step(to_batch([train_graph(CBY[c], LABELS[c]) for c in cids]))
    (ld["loss"] * weight).backward()
    return float(ld["loss"].detach())


MODEL.train()
n_pilot = min(PILOT_CASES, len(TRAIN_IDS))
t0 = time.perf_counter()
for i in range(0, n_pilot, BATCH):
    train_step(TRAIN_IDS[i:i + BATCH], 1.0 / ACCUM)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
SEC_PER_CASE = (time.perf_counter() - t0) / n_pilot
# the response weight: released-model published-objective loss on the pilot cases over its response loss on as many pairs
MODEL.eval()
with torch.no_grad():
    l_obj = [float(TASK.shared_step(to_batch([train_graph(CBY[c], LABELS[c]) for c in TRAIN_IDS[i:i + BATCH]]))[1]["loss"]) for i in range(0, n_pilot, BATCH)]
    pair_pilot = np.random.default_rng(SEED + 6).choice(PAIR_TRAIN, n_pilot, replace=False).tolist()
    l_resp = [float(response_loss(pair_pilot[i:i + PAIRS_PER_STEP])) for i in range(0, n_pilot, PAIRS_PER_STEP)]
MODEL.train()
RESPONSE_WEIGHT = float(min(MAX_RESPONSE_WEIGHT, np.mean(l_obj) / np.mean(l_resp)))
print(f"released model: published objective {np.mean(l_obj):.5f}, response loss {np.mean(l_resp):.5f} -> response weight {RESPONSE_WEIGHT:.4g}")
opt.zero_grad(set_to_none=True)
MODEL.load_state_dict(RELEASED_STATE)                   # the pilot's gradients are discarded; no step was taken
STEPS_PER_EPOCH = math.ceil(len(TRAIN_IDS) / (BATCH * ACCUM))
CASES_PER_EPOCH = len(TRAIN_IDS) + STEPS_PER_EPOCH * 2 * PAIRS_PER_STEP
EPOCHS = int(min(MAX_EPOCHS, max(1, math.floor(TRAIN_BUDGET_MIN * 60 / (SEC_PER_CASE * CASES_PER_EPOCH)))))
SIZING = {"pilot_cases": n_pilot, "seconds_per_case": SEC_PER_CASE, "training_cases": len(TRAIN_IDS), "epochs": EPOCHS,
          "optimizer_steps": EPOCHS * STEPS_PER_EPOCH, "pairs_per_step": PAIRS_PER_STEP, "response_weight": RESPONSE_WEIGHT,
          "released_pilot_objective": float(np.mean(l_obj)), "released_pilot_response_loss": float(np.mean(l_resp)),
          "accumulation": "loss of each micro-batch weighted by its share of the cases of its optimizer step; the last step of an epoch takes the remainder; no gradient carried across epochs"}
print(f"pilot: {SEC_PER_CASE * 1000:.0f} ms per case (graph, forward, backward) -> {EPOCHS} epoch(s), {SIZING['optimizer_steps']} optimizer steps, "
      f"about {SEC_PER_CASE * CASES_PER_EPOCH * EPOCHS / 60:.0f} min")

In [ ]:
def validate():
    MODEL.eval()
    losses, err = [], {}
    with torch.no_grad():
        for i in range(0, len(VAL_IDS), VAL_BATCH):
            cids = VAL_IDS[i:i + VAL_BATCH]
            b = to_batch([train_graph(CBY[c], LABELS[c]) for c in cids])
            out, ld = TASK.shared_step(b)
            losses.append(float(ld["loss"]) * len(cids))
            vm = out["bus"][:, VM_OUT].float().cpu().numpy(); ptr = b["bus"].ptr.cpu().numpy()
            for j, c in enumerate(cids):
                d = BASE[CBY[c]["base"]]
                e = np.abs(vm[ptr[j]:ptr[j + 1]] - LABELS[c][0])[d["PQ"]]
                err.setdefault(VAL_GROUP[c], []).append(float(e.max()))
        slopes = []                                                # load-growth response on the monitoring pairs
        for i in range(0, len(PAIR_VAL), VAL_BATCH):
            cids = PAIR_VAL[i:i + VAL_BATCH]
            bb = to_batch([train_graph(CBY[c], LABELS[c]) for c in cids]); bg = to_batch([pair_graph(CBY[c], PAIR_LABELS[c]) for c in cids])
            vb = forward(MODEL, bb)["bus"][:, VM_OUT].float().cpu().numpy(); vg = forward(MODEL, bg)["bus"][:, VM_OUT].float().cpu().numpy()
            ptr = bb["bus"].ptr.cpu().numpy()
            for j, c in enumerate(cids):
                pq = BASE[CBY[c]["base"]]["PQ"]
                g_ = (vg[ptr[j]:ptr[j + 1]] - vb[ptr[j]:ptr[j + 1]])[pq] / GROWTH
                t_ = ((PAIR_LABELS[c][0] - LABELS[c][0]) / GROWTH)[pq]
                xt, yg = t_ - t_.mean(), g_ - g_.mean()
                slopes.append(float(xt @ yg / (xt @ xt)) if xt @ xt > 0 else np.nan)
    MODEL.train()
    res = {"val_loss": sum(losses) / len(VAL_IDS), "response_slope_median": float(np.nanmedian(slopes))}
    for g_, v in err.items():
        res[f"worst_bus_err_median_{g_}"] = float(np.median(v))
    return res


LOG = []
v0 = validate(); LOG.append({"step": 0, "epoch": 0, "train_loss": np.nan, **v0})
print("released model on the validation split:", {k: round(v, 5) for k, v in v0.items()})
best_loss, best_step = v0["val_loss"], 0
BEST_STATE = copy.deepcopy(RELEASED_STATE)
rng_t = np.random.default_rng(SEED + 2 + 1000 * TRAIN_SEED)
rng_p = np.random.default_rng(SEED + 3 + 1000 * TRAIN_SEED)
pair_order = []


def next_pairs():
    """The next PAIRS_PER_STEP training pairs: passes through the pool in a fresh random order each time."""
    global pair_order
    if len(pair_order) < PAIRS_PER_STEP:
        pair_order = pair_order + rng_p.permutation(PAIR_TRAIN).tolist()
    mb, pair_order = pair_order[:PAIRS_PER_STEP], pair_order[PAIRS_PER_STEP:]
    return mb


resp_loss = []
torch.manual_seed(TRAIN_SEED)
step, t_start, run_loss = 0, time.perf_counter(), []
MODEL.train(); opt.zero_grad(set_to_none=True)
for ep in range(1, EPOCHS + 1):
    order = rng_t.permutation(TRAIN_IDS).tolist()
    # optimizer steps of ACCUM micro-batches; the last step of the epoch takes whatever remains, so no gradient is ever carried
    # into the next epoch and every case of the epoch is used exactly once with equal weight
    chunks = [order[i:i + BATCH * ACCUM] for i in range(0, len(order), BATCH * ACCUM)]
    for chunk in chunks:
        for j in range(0, len(chunk), BATCH):
            mb = chunk[j:j + BATCH]
            run_loss.append(train_step(mb, len(mb) / len(chunk)))
        resp_loss.append(response_step(next_pairs(), RESPONSE_WEIGHT))
        if True:
            for g_ in opt.param_groups:
                g_["lr"] = LR * min(1.0, (step + 1) / WARMUP_STEPS)
            torch.nn.utils.clip_grad_norm_(MODEL.parameters(), CLIP)
            opt.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % VAL_EVERY == 0:
                v = validate(); LOG.append({"step": step, "epoch": ep, "train_loss": float(np.mean(run_loss)), "response_loss": float(np.mean(resp_loss)), **v}); run_loss, resp_loss = [], []
                if v["val_loss"] < best_loss:
                    best_loss, best_step = v["val_loss"], step
                    BEST_STATE = copy.deepcopy({k: t.detach().cpu().clone() for k, t in MODEL.state_dict().items()})
                print(f"  epoch {ep} step {step}: train {LOG[-1]['train_loss']:.5f} | val {v['val_loss']:.5f} | "
                      + " | ".join(f"{k[21:]} {v_:.4f}" for k, v_ in v.items() if k.startswith("worst")) + f" | {(time.perf_counter() - t_start) / 60:.0f} min")
    v = validate(); LOG.append({"step": step, "epoch": ep, "train_loss": float(np.mean(run_loss)) if run_loss else np.nan, **v}); run_loss = []
    if v["val_loss"] < best_loss:
        best_loss, best_step = v["val_loss"], step
        BEST_STATE = copy.deepcopy({k: t.detach().cpu().clone() for k, t in MODEL.state_dict().items()})
    print(f"end of epoch {ep}: val {v['val_loss']:.5f} (best {best_loss:.5f} at step {best_step})")
TRAIN_MIN = (time.perf_counter() - t_start) / 60
LOGDF = pd.DataFrame(LOG); LOGDF.to_csv(Path(OUT_DIR) / "training_log.csv", index=False)
print(f"training: {step} optimizer steps in {TRAIN_MIN:.0f} min; selected step {best_step} (validation loss {best_loss:.5f} against {v0['val_loss']:.5f} released)")

## 6. The selected checkpoint and output files

In [ ]:
import hashlib
ck = Path(OUT_DIR) / "genco_finetuned_state_dict.pt"
torch.save(BEST_STATE, ck)
CK_SHA = hashlib.sha256(ck.read_bytes()).hexdigest()
MODEL.load_state_dict(BEST_STATE); MODEL.eval()
vb = validate()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axes[0]
ax.plot(LOGDF.step, LOGDF.val_loss, "o-", label="validation loss"); ax.plot(LOGDF.step, LOGDF.train_loss, ".-", label="training loss (running)")
ax.axvline(best_step, color="k", ls=":", lw=1); ax.set_xlabel("optimizer step"); ax.set_ylabel("released objective"); ax.legend(fontsize=8); ax.set_yscale("log")
ax = axes[1]
for col in [c_ for c_ in LOGDF.columns if c_.startswith("worst")]:
    ax.plot(LOGDF.step, LOGDF[col], "o-", label=col[21:])
ax.axvline(best_step, color="k", ls=":", lw=1); ax.set_xlabel("optimizer step"); ax.set_ylabel("worst-bus |Vm error|, median (p.u.)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(Path(OUT_DIR) / "fig_training.png", dpi=150); plt.show()

In [ ]:
import platform, shutil
summary = {"frozen_protocol": FT_PROTOCOL, "sizing": SIZING, "training_minutes": TRAIN_MIN, "optimizer_steps": step, "selected_step": best_step,
           "validation_released": v0, "validation_selected": vb, "checkpoint_sha256": CK_SHA,
           "load_growth": {"growth": GROWTH, "pairs_train": len(PAIR_TRAIN), "pairs_monitor": len(PAIR_VAL), "pairs_per_step": PAIRS_PER_STEP,
                           "response_weight": RESPONSE_WEIGHT, "response_slope_released": v0["response_slope_median"],
                           "response_slope_selected": vb["response_slope_median"]},
           "cases": {"built": len(CASES), "unresolved": int((TC.status == "unresolved").sum()), "multiple_solutions": int(TC.multiple_solutions.sum()),
                     "regression_targets_train": len(TRAIN_IDS), "regression_targets_val": len(VAL_IDS_ALL)},
           "graphkit": {"checked_out_commit": subprocess.run(["git", "-C", GRAPHKIT_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()},
           "versions": {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__, "device": str(DEVICE),
                        "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none", "lightsim2grid": HAVE_KLU}}
json.dump(summary, open(Path(OUT_DIR) / "summary.json", "w"), indent=1, default=str)
print("VERDICT")
print(f"- {len(CASES)} cases on {len(BASE_IDS)} operating points; {summary['cases']['unresolved']} unresolved, {summary['cases']['multiple_solutions']} with two solutions; "
      f"{len(TRAIN_IDS)} training targets, {len(VAL_IDS_ALL)} validation targets.")
print(f"- Load-growth pairs: {len(PAIR_TRAIN)} for training, weight {RESPONSE_WEIGHT:.4g}; response slope on the monitoring pairs "
      f"{v0['response_slope_median']:.2f} (released) -> {vb['response_slope_median']:.2f} (selected).")
print(f"- Validation loss: released {v0['val_loss']:.5f} -> selected {vb['val_loss']:.5f} (step {best_step} of {step}).")
for k in [k for k in vb if k.startswith("worst")]:
    print(f"- {k}: released {v0.get(k, float('nan')):.4f} -> fine-tuned {vb[k]:.4f} p.u.")
print(f"- Checkpoint SHA-256 {CK_SHA[:16]}...  This is a validation result only; the test is notebook evaluate_finetuned.")
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"written {OUT_DIR}.zip ({os.path.getsize(OUT_DIR + '.zip') / 1e6:.0f} MB); evaluate_finetuned needs the checkpoint inside it.")
save_to_drive(OUT_DIR + ".zip")